In [1]:
import pandas as pd
import numpy as np

url = ("https://raw.githubusercontent.com/jxchen/Kaggle"
       "/master/Give%20Me%20Some%20Credit/cs-training.csv")
df_raw = pd.read_csv(url, index_col=0)
df = df_raw.rename(columns={
    "SeriousDlqin2yrs":                    "vo_no",
    "RevolvingUtilizationOfUnsecuredLines": "ty_le_su_dung_tin_dung",
    "age":                                  "tuoi",
    "NumberOfTime30-59DaysPastDueNotWorse": "so_lan_tre_30_59_ngay",
    "DebtRatio":                            "ty_le_no",
    "MonthlyIncome":                        "thu_nhap_thang",
    "NumberOfOpenCreditLinesAndLoans":      "so_tai_khoan_vay",
    "NumberOfTimes90DaysLate":              "so_lan_tre_90_ngay",
    "NumberRealEstateLoansOrLines":         "so_tai_khoan_bat_dong_san",
    "NumberOfTime60-89DaysPastDueNotWorse": "so_lan_tre_60_89_ngay",
    "NumberOfDependents":                   "so_nguoi_phu_thuoc"
})
print(f"Shape: {df.shape} | NaN: {df.isnull().sum().sum()}")

Shape: (150000, 11) | NaN: 33655


In [4]:
# agg — output là bảng tóm tắt
result = df.groupby("vo_no").agg(
    so_luong        = ("tuoi", "count"),
    tuoi_tb         = ("tuoi", "mean"),
    thu_nhap_median = ("thu_nhap_thang", "median"),
    ty_le_su_dung_tin_dung_tb = ("ty_le_su_dung_tin_dung", "median"),
    so_lan_tre_tb   = ("so_lan_tre_90_ngay", "mean")
).round(3)
print(result)

       so_luong  tuoi_tb  thu_nhap_median  ty_le_su_dung_tin_dung_tb  \
vo_no                                                                  
0        139974   52.751           5466.0                      0.133   
1         10026   45.927           4500.0                      0.839   

       so_lan_tre_tb  
vo_no                 
0              0.135  
1              2.091  


In [5]:
nhom_tuoi = pd.cut(df["tuoi"],
       bins=[0, 30, 45, 60, 120],    # ranh giới các khoảng
       labels=["<30", "30-45", "45-60", "60+"])  # tên từng khoảng
df["nhom_tuoi"] = df.groupby("vo_no")["tuoi"].transform("mean")
print(df[["vo_no", "tuoi", "nhom_tuoi"]].head())

   vo_no  tuoi  nhom_tuoi
1      1    45  45.926591
2      0    40  52.751375
3      0    38  52.751375
4      0    30  52.751375
5      0    49  52.751375


In [7]:
df["nhom_tuoi"] = pd.cut(df["tuoi"],
                         bins=[0, 30, 45, 60, 120],
                         labels=["<30", "30-45", "45-60", "60+"])

df["ty_le_vo_no_nhom_tuoi"] = df.groupby("nhom_tuoi", observed=True)["vo_no"].transform("mean")

# In tỷ lệ vỡ nợ 4 nhóm
print(df.groupby("nhom_tuoi", observed=True)["vo_no"].mean().round(3))

nhom_tuoi
<30      0.116
30-45    0.093
45-60    0.068
60+      0.030
Name: vo_no, dtype: float64


In [3]:
df["nhom_tre_han"] = df["so_lan_tre_90_ngay"].clip(upper=5)
print(df.groupby("nhom_tre_han")["vo_no"].mean().round(3))

nhom_tre_han
0    0.046
1    0.337
2    0.499
3    0.577
4    0.670
5    0.603
Name: vo_no, dtype: float64


In [8]:
# Weighted mean của ty_le_no, trọng số = thu_nhap_thang
# Loại NaN trước
df_clean = df[["nhom_tuoi", "ty_le_no", "thu_nhap_thang"]].dropna()

weighted_mean = (
    df_clean
    .groupby("nhom_tuoi", observed=True)
    .apply(lambda g: (g["ty_le_no"] * g["thu_nhap_thang"]).sum()
                      / g["thu_nhap_thang"].sum(),
           include_groups=False)
    .round(3)
)

# So sánh với mean thông thường
simple_mean = df_clean.groupby("nhom_tuoi", observed=True)["ty_le_no"].mean().round(3)

print(pd.DataFrame({"weighted_mean": weighted_mean, "simple_mean": simple_mean}))

           weighted_mean  simple_mean
nhom_tuoi                            
<30                0.215       13.937
30-45              0.338       30.652
45-60              0.339       28.749
60+                0.278       22.916
